# 0. 경로설정

In [1]:
BASE_DIR = "lerobot_project"
VENV_NAME = "lerobot_env"
PYTHON_VERSION = "3.12"
REPO_NAME = "lerobot-ac-mlp"

import os, sys, subprocess
os.environ['MPLBACKEND'] = 'Agg'
home_dir = os.path.expanduser('~')
base_path = os.path.join(home_dir, BASE_DIR)
venv_path = os.path.join(base_path, VENV_NAME)
site_packages = os.path.join(venv_path, 'lib', f'python{PYTHON_VERSION}', 'site-packages')
if site_packages not in sys.path: sys.path.insert(0, site_packages)
python_exec = os.path.join(venv_path, 'bin', 'python')
pip_exec = os.path.join(venv_path, 'bin', 'pip')
project_dir = os.path.join(base_path, REPO_NAME)
src_path = os.path.join(project_dir, 'src')
if src_path not in sys.path: sys.path.insert(0, src_path)

# --- 렌더링 환경 설정 (osmesa) ---
conda_lib = os.path.join(home_dir, 'miniconda3', 'lib')
lib_dirs = {conda_lib}
r = subprocess.run(['bash', '-c', f'find {home_dir}/miniconda3 -name "libOSMesa*" -o -name "libEGL*" 2>/dev/null'],
                    capture_output=True, text=True, timeout=15)
for line in r.stdout.strip().split('\n'):
    if line.strip(): lib_dirs.add(os.path.dirname(line.strip()))
ld_path = ':'.join(lib_dirs) + ':' + os.environ.get('LD_LIBRARY_PATH', '')
os.environ['LD_LIBRARY_PATH'] = ld_path
os.environ['MUJOCO_GL'] = 'osmesa'

%cd {project_dir}
print(f"Project: {project_dir}")
print(f"Rendering: {os.environ['MUJOCO_GL']}")
!git branch --show-current

/home1/eunji24/lerobot_project/lerobot-ac-mlp
Project: /home1/eunji24/lerobot_project/lerobot-ac-mlp
Rendering: osmesa
/bin/bash: /home1/eunji24/miniconda3/lib/libtinfo.so.6: no version information available (required by /bin/bash)
ecd-eunji


# 1.1 ACM eval (baseline)

In [ ]:
import os, subprocess, sys

cmd = [
    python_exec,
    "-m", "lerobot.scripts.lerobot_eval",
    "--policy.path=outputs/train/acm_ens_test_100k/checkpoints/080000/pretrained_model",
    "--policy.n_action_steps=80",
    "--policy.chunk_size=150", 
    "--env.type=aloha",
    "--env.task=AlohaTransferCube-v0",
    "--eval.n_episodes=200",
    "--eval.batch_size=1",
    "--output_dir=outputs/eval/ens_base_epi",
    
]

env = os.environ.copy()
env["MUJOCO_GL"] = os.environ["MUJOCO_GL"]
env["LD_LIBRARY_PATH"] = os.environ["LD_LIBRARY_PATH"]

print("CMD:", " ".join(cmd))

proc = subprocess.Popen(
    cmd,
    env=env,
    stdout=subprocess.PIPE,
    stderr=subprocess.STDOUT,
    text=True,
    bufsize=1,
)

for line in proc.stdout:
    print(line, end="")

proc.wait()
print("\nRETURN CODE:", proc.returncode)

CMD: /home1/eunji24/lerobot_project/lerobot_env/bin/python -m lerobot.scripts.lerobot_eval --policy.path=outputs/train/acm_ens_test_100k/checkpoints/080000/pretrained_model --policy.n_action_steps=80 --policy.chunk_size=150 --env.type=aloha --env.task=AlohaTransferCube-v0 --eval.n_episodes=200 --eval.batch_size=1 --output_dir=outputs/eval/ens_base_epi


# 1.2 ACM eval (Temporal ensemble coeff)

In [ ]:
import os, subprocess, sys

cmd = [
    python_exec,
    "-m", "lerobot.scripts.lerobot_eval",
    "--policy.path=outputs/train/acm_transfer_cube_test_100k/checkpoints/080000/pretrained_model",
    "--policy.n_action_steps=1",
    "--policy.chunk_size=100", 
    "--policy.temporal_ensemble_coeff=0.01",
    "--env.type=aloha",
    "--env.task=AlohaTransferCube-v0",
    "--eval.n_episodes=50",
    "--eval.batch_size=1",
    "--output_dir=outputs/eval/acm_temporal_001",
    
]

env = os.environ.copy()
env["MUJOCO_GL"] = os.environ["MUJOCO_GL"]
env["LD_LIBRARY_PATH"] = os.environ["LD_LIBRARY_PATH"]

print("CMD:", " ".join(cmd))

proc = subprocess.Popen(
    cmd,
    env=env,
    stdout=subprocess.PIPE,
    stderr=subprocess.STDOUT,
    text=True,
    bufsize=1,
)

for line in proc.stdout:
    print(line, end="")

proc.wait()
print("\nRETURN CODE:", proc.returncode)

CMD: /home1/eunji24/lerobot_project/lerobot_env/bin/python -m lerobot.scripts.lerobot_eval --policy.path=outputs/train/acm_transfer_cube_test_100k/checkpoints/080000/pretrained_model --policy.n_action_steps=1 --policy.chunk_size=100 --policy.temporal_ensemble_coeff=0.01 --env.type=aloha --env.task=AlohaTransferCube-v0 --eval.n_episodes=50 --eval.batch_size=1 --output_dir=outputs/eval/acm_temporal_001
WARNING 2026-04-13 20:55:03 figs/eval.py:63 No job name provided, using 'aloha_acm' as job name.
INFO 2026-04-13 20:55:03 bot_eval.py:556 {'env': {'disable_env_checker': True,
         'episode_length': 400,
         'features': {'action': {'shape': (14,),
                                 'type': <FeatureType.ACTION: 'ACTION'>},
                      'agent_pos': {'shape': (14,),
                                    'type': <FeatureType.STATE: 'STATE'>},
                      'pixels/top': {'shape': (480, 640, 3),
                                     'type': <FeatureType.VISUAL: 'VISUAL'>}}